In [2]:
#  CC3074 – Minería de Datos
#  Proyecto 3. Avances 2. Antecedentes y selección de algoritmos
# 
#  Integrantes del grupo:
#  Diego Sandoval - 231977
#  Jorge Gabriel Palacios Sales - 231385
#  Anggelie Lizeth Velásquez Asencio - 221181
#  Roberto Camposeco Torres – 23968 

In [3]:
# Imports globales compartidos por todos los módulos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
SEED = 42

In [4]:

# Configuración de las rutas

BASE_DIR = os.getcwd()

RUTAS = {
    'MATRIMONIOS': os.path.join(BASE_DIR, 'data_matrimonios'),
    'DIVORCIOS':   os.path.join(BASE_DIR, 'data_divorcios')
}

MAPA_COLUMNAS = {
    'A_OCUR':   'ANIO_OCURRENCIA',
    'MES_OCUR': 'MES_OCURRENCIA',
    'DEPTO':    'DEPARTAMENTO',
    'MUN':      'MUNICIPIO',
    'EDADHOM':  'EDAD_HOMBRE',
    'EDADMUJ':  'EDAD_MUJER',
}

VALORES_INVALIDOS = [99, 999, 9999, 98, 998, 9998]

for k, v in RUTAS.items():
    existe = 'CHECK' if os.path.exists(v) else ' NO ENCONTRADA'
    print(f'   {k}: {v} {existe}')

   MATRIMONIOS: c:\Users\angge\mineria\CorteProyecto1Mineria\data_matrimonios CHECK
   DIVORCIOS: c:\Users\angge\mineria\CorteProyecto1Mineria\data_divorcios CHECK


In [5]:
#Función para cargar datasets INE

def cargar_dataset_ine(ruta_carpeta, nombre_dataset):
    print(f'\n{"="*55}')
    print(f'  Cargando: {nombre_dataset}')
    print(f'{"="*55}')

    if not os.path.exists(ruta_carpeta):
        print(f' Ruta no encontrada: {ruta_carpeta}')
        return None

    archivos_sav = sorted([f for f in os.listdir(ruta_carpeta) if f.endswith('.sav')])

    if not archivos_sav:
        print(f' No se encontraron archivos .sav')
        return None

    lista_dfs = []
    for archivo in archivos_sav:
        full_path = os.path.join(ruta_carpeta, archivo)
        try:
            df_temp = pd.read_spss(full_path, convert_categoricals=False)
            df_temp.columns = df_temp.columns.str.strip().str.upper()
            df_temp.rename(columns=MAPA_COLUMNAS, inplace=True)
            df_temp['ARCHIVO_ORIGEN'] = archivo
            lista_dfs.append(df_temp)
            print(f' {archivo} ({len(df_temp):,} filas)')
        except Exception as e:
            print(f'Error: {archivo}: {e}')

    if not lista_dfs:
        return None

    df = pd.concat(lista_dfs, ignore_index=True)
    df.replace(VALORES_INVALIDOS, np.nan, inplace=True)
    print(f'\n Total: {len(df):,} filas | {df.shape[1]} columnas')
    return df

print('Función cargar_dataset_ine definida')

Función cargar_dataset_ine definida


In [6]:


#wwaLimpieza y variables derivadas


def limpiar_edades(df):
    for col in ['EDAD_HOMBRE', 'EDAD_MUJER']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df.loc[(df[col] < 14) | (df[col] > 100), col] = np.nan
    return df

def crear_variables_derivadas(df):
    if 'EDAD_HOMBRE' in df.columns and 'EDAD_MUJER' in df.columns:
        df['DIFERENCIA_EDAD'] = (df['EDAD_HOMBRE'] - df['EDAD_MUJER']).abs()
        df['EDAD_PROMEDIO']   = (df['EDAD_HOMBRE'] + df['EDAD_MUJER']) / 2

        bins   = [0, 24, 34, 44, 60, 100]
        labels = ['<25', '25-34', '35-44', '45-60', '60+']
        df['SEGMENTO_EDAD_HOM'] = pd.cut(df['EDAD_HOMBRE'], bins=bins, labels=labels, right=False)

        df['BRECHA_EDAD_CAT'] = pd.cut(
            df['DIFERENCIA_EDAD'],
            bins=[-1, 2, 5, 10, 100],
            labels=['Pequeña (0-2)', 'Media (3-5)', 'Grande (6-10)', 'Muy grande (10+)']
        )
    return df

print('Funciones de limpieza y variables derivadas definidas')

Funciones de limpieza y variables derivadas definidas


In [7]:
# Construcción de df_master
df_matrimonios = cargar_dataset_ine(RUTAS['MATRIMONIOS'], 'MATRIMONIOS')
df_divorcios   = cargar_dataset_ine(RUTAS['DIVORCIOS'],   'DIVORCIOS')

if df_matrimonios is not None and df_divorcios is not None:
    df_matrimonios['DIVORCIO'] = 0
    df_divorcios['DIVORCIO']   = 1

    df_matrimonios = limpiar_edades(df_matrimonios)
    df_divorcios   = limpiar_edades(df_divorcios)

    df_master = pd.concat([df_matrimonios, df_divorcios], ignore_index=True)
    df_master = crear_variables_derivadas(df_master)

    print(f'\n{"="*55}')
    print('  REPORTE DE CALIDAD — df_master')
    print(f'{"="*55}')
    print(f'  Filas   : {len(df_master):,}')
    print(f'  Columnas: {df_master.shape[1]}')
    print(f'\n  Variable DIVORCIO:')
    print(df_master['DIVORCIO'].value_counts(normalize=True).mul(100).round(2).to_string())

    nulos = df_master.isnull().sum()
    nulos = nulos[nulos > 0].sort_values(ascending=False).head(10)
    if not nulos.empty:
        print(f'\n  Columnas con nulos (top 10):')
        print(nulos.to_string())

    print('\ndf_master construido exitosamente')
else:
    print('Error: no se pudo construir df_master')


  Cargando: MATRIMONIOS
 20191129151512xADe9MdA1kzkD05EfolYPKJsBEP23S1S.sav (74,777 filas)
 20201201155014FaXwFKh8NYNiFivgBo98JEbaVMRUhaFG.sav (76,928 filas)
 20210730183005QL4dMFKDwxMZkfk1gTiHClmLsdYdyRgL.sav (57,387 filas)
 20220729171110puPW7O9wJalS7I9yToxkpQLgLwNElAny.sav (87,480 filas)
 20230728221653mVWH0fGZodIVidGdKRwl4tU3OHKjKnen (1).sav (80,875 filas)
 20230728221653mVWH0fGZodIVidGdKRwl4tU3OHKjKnen.sav (80,875 filas)
 4SRVVUxXZXkoQGnZKjH4bYaW8tPyYdhQ (1).sav (79,177 filas)
 4SRVVUxXZXkoQGnZKjH4bYaW8tPyYdhQ (2).sav (79,177 filas)
 4SRVVUxXZXkoQGnZKjH4bYaW8tPyYdhQ.sav (79,177 filas)

 Total: 695,853 filas | 24 columnas

  Cargando: DIVORCIOS
 2W4N7bnfbaVntLsINTDXCUZ57oCLyC3o (1).sav (5,726 filas)
 2W4N7bnfbaVntLsINTDXCUZ57oCLyC3o (2).sav (5,726 filas)
 2W4N7bnfbaVntLsINTDXCUZ57oCLyC3o (3).sav (5,726 filas)
 2W4N7bnfbaVntLsINTDXCUZ57oCLyC3o (4).sav (5,726 filas)
 2W4N7bnfbaVntLsINTDXCUZ57oCLyC3o (5).sav (5,726 filas)
 2W4N7bnfbaVntLsINTDXCUZ57oCLyC3o (6).sav (5,726 filas)
 2W4N7

In [9]:
# Estadísticas descriptivas
print('EDA MATRIMONIOS — Estadísticas Descriptivas\n')

cols_interes = [c for c in ['EDAD_HOMBRE', 'EDAD_MUJER', 'DIFERENCIA_EDAD',
                             'EDAD_PROMEDIO', 'MESREG', 'DEPREG', 'AÑOREG']
                if c in df_matrimonios.columns]

resumen = df_matrimonios[cols_interes].describe().T
resumen['skewness'] = df_matrimonios[cols_interes].skew()
resumen['kurtosis'] = df_matrimonios[cols_interes].kurt()
display(resumen[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max', 'skewness', 'kurtosis']].round(2))

EDA MATRIMONIOS — Estadísticas Descriptivas



,count,mean,std,min,25%,50%,75%,max,skewness,kurtosis
EDAD_HOMBRE,695781.0,29.91,11.33,14.0,22.0,26.0,33.0,97.0,1.93,4.18
EDAD_MUJER,695709.0,26.94,9.92,14.0,20.0,24.0,30.0,95.0,1.97,4.76
MESREG,695853.0,6.52,3.59,1.0,3.0,6.0,10.0,12.0,0.02,-1.29
DEPREG,695853.0,9.64,6.37,1.0,4.0,10.0,14.0,22.0,0.11,-1.12
AÑOREG,695853.0,2018.65,2.85,2015.0,2015.0,2019.0,2021.0,2023.0,-0.17,-1.55


In [ ]:
#  Distribución de edades

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribución de Edades en Matrimonios', fontsize=14, fontweight='bold')

for ax, col, color, label in zip(
    axes,
    ['EDAD_HOMBRE', 'EDAD_MUJER'],
    ['steelblue', 'salmon'],
    ['Hombre', 'Mujer']
):
    if col in df_matrimonios.columns:
        data = df_matrimonios[col].dropna()
        ax.hist(data, bins=40, color=color, alpha=0.7, edgecolor='white', density=True)
        data.plot.kde(ax=ax, color='black', linewidth=1.5)
        ax.axvline(data.mean(),   color='red',   linestyle='--', label=f'Media: {data.mean():.1f}')
        ax.axvline(data.median(), color='green', linestyle='--', label=f'Mediana: {data.median():.1f}')
        ax.set_title(f'Edad del {label}')
        ax.set_xlabel('Edad')
        ax.legend()

plt.tight_layout()
plt.show()